# LLM-as-Judge & Evaluation

## Why this matters

This is the missing piece from the entire X++ project's reproducibility work. Everything done there -- moving checks to deterministic Python, `temperature=0`, tool-use schemas -- measured and improved *consistency* (does the same input produce the same output). None of it measured *correctness* (is the output actually right). The project's own README states this directly: "No ground-truth validation... Security/Performance findings are Claude's assessment, not verified against a compiler or a labeled vulnerability dataset." LLM-as-judge is the real technique that exists for exactly this gap -- not a replacement for the compiler-based ground truth that's genuinely blocked for six months, but a way to get *some* signal on correctness in the meantime, which the project currently has none of.

Note: this is deliberately scoped to general-purpose evaluation, not RAG-specific evaluation -- RAGAS and retrieval-specific evaluation metrics are covered separately in `rag-and-retrieval/06-rag-evaluation-ragas`, since that's a distinct (though related) topic with its own dedicated metrics (faithfulness, context precision/recall) that don't apply outside a retrieval pipeline.

## Level 1: What it is

**LLM-as-judge**: using an LLM call to *evaluate* another LLM's output, typically against a rubric, rather than relying only on human review or hard-coded assertions. Three common shapes:

1. **Pairwise comparison** -- given two candidate outputs for the same input, the judge picks which is better (or ties), often used to compare two prompt versions or two models against each other.
2. **Pointwise scoring** -- the judge scores a single output against a rubric (e.g., 1-5 on "is this security finding well-justified"), without needing a second output to compare against.
3. **Reference-based grading** -- the judge compares an output against a known-correct reference answer, checking for factual/semantic equivalence rather than exact string match (useful when the correct answer can be phrased many valid ways).

## Level 2: How it actually works internally

### Why not just use exact-match assertions?

For a lot of software, testing means asserting exact output: `assert result == expected`. This works for the X++ project's *deterministic* checks (which is exactly why they're testable the way they were -- the synthetic-file tests that caught the `while select` and `Box::info()` bugs used plain equality checks, because deterministic output has one correct answer). It fundamentally does not work for the LLM-judgment findings, because there's no single correct phrasing of "this authorization check is missing" -- two genuinely correct findings can use completely different wording, different levels of detail, different but equally valid severity framing. Exact match would fail both a genuinely correct and a genuinely wrong finding for the wrong reason (wording), which is useless as a signal.

### The actual mechanism

A judge prompt typically supplies: the original input, the candidate output being evaluated, a rubric or set of criteria, and a request for a structured verdict (often via tool-use, tying directly back to the structured-output notebook -- you want the judge's verdict itself to be reliably parseable, same reasoning as `base_agent.py`'s `ISSUE_TOOL`). For the X++ project specifically, a judge call might look like:

```python
JUDGE_TOOL = {
    "name": "grade_finding",
    "input_schema": {
        "type": "object",
        "properties": {
            "is_valid_finding": {"type": "boolean"},
            "severity_appropriate": {"type": "boolean"},
            "reasoning": {"type": "string"},
        },
        "required": ["is_valid_finding", "severity_appropriate", "reasoning"],
    },
}
# judge is given: the X++ source code + one specific Security finding
# judge is asked: is this a real issue given the code, and is Critical
# the right severity, or should it be Major?
```

Run this judge across the three repeated runs' worth of findings from the X++ sessions (the ones where the container-ownership-check finding appeared in 2 of 3 runs), and you'd get an *independent* second opinion on which of those runs was actually right -- rather than just knowing they disagreed, which was as far as the manual comparison went.

## Level 3: Where it breaks, and what it doesn't solve

This is the section that matters most for this topic, because LLM-as-judge has real, well-documented failure modes that are easy to gloss over:

1. **Self-preference bias.** A model tends to rate its own outputs (or outputs from the same model family) more favorably than outputs from a different model, even when quality is genuinely comparable. If the judge is also Claude Sonnet, and the thing being judged was also generated by Claude Sonnet, there's a documented tendency toward inflated scores relative to a neutral third party. Mitigation: use a different, typically stronger model as judge than the model being evaluated, when feasible -- or explicitly account for this bias when interpreting results rather than treating the judge's verdict as ground truth.

2. **Verbosity bias.** Judges (like human raters, this isn't unique to LLMs) tend to rate longer, more detailed responses as "better" even when length doesn't correlate with correctness. A `suggested_fix` field that's three paragraphs isn't automatically more correct than one that's two sentences -- but a judge prompt that doesn't explicitly control for this can end up measuring verbosity, not quality.

3. **Position bias in pairwise comparison.** When shown two outputs side by side, judges show a measurable tendency to favor whichever one is presented first (or, on some setups, second) -- an artifact of the comparison setup, not the outputs' actual quality. Mitigation: run each pairwise comparison twice with the order swapped, and only trust a verdict that holds in both orders.

4. **The judge itself has no ground truth guarantee -- this is the meta-problem, and it's the one that actually matters most for the X++ project.** An LLM-as-judge system doesn't magically produce ground truth; it produces a *second, independent model opinion*, which is useful (agreement between two independent judgments is stronger evidence than one judgment alone) but is not the same thing as compiler-verified correctness. For the X++ project specifically: LLM-as-judge could be a genuinely useful *interim* signal while environment access is blocked -- e.g., "do two independent judge calls agree this authorization-check finding is valid" is more evidence than one agent's single pass -- but it cannot substitute for the actual AOT-compiled verification that's the real, only source of ground truth for whether a security finding is *correct* rather than merely *plausible-sounding*. Don't let LLM-as-judge quietly become the thing that makes the "no ground truth" limitation feel solved when it isn't -- it narrows the gap, it doesn't close it.

5. **Judges need their own calibration against human judgment before you trust them.** The standard practice: have a human expert grade a sample of outputs, have the LLM judge grade the same sample, and measure agreement (e.g., Cohen's kappa or simple percent agreement) before trusting the judge on the full dataset. Skipping this step means you don't actually know if the judge's opinions track a real expert's opinions or just sound authoritative -- which is a real risk given how confidently LLMs phrase things regardless of actual certainty.

6. **Cost and latency compound.** Every judged output now costs an *additional* LLM call on top of the original generation -- for a system already running four parallel review agents, adding a judge pass roughly doubles the API cost for whatever's being evaluated. Worth scoping judge evaluation to a sample (e.g., judge the flaky finding specifically, not every finding from every agent) rather than judging everything by default.

In [ ]:
# Concrete sketch: using LLM-as-judge specifically on the flaky Security
# finding from the X++ sessions (container-ownership/authorization check,
# ~66% catch rate across 3 real runs). This doesn't fix the underlying
# no-ground-truth problem, but gives an independent second opinion on
# whether the finding is valid when it DOES appear.

from anthropic import Anthropic

client = Anthropic()

JUDGE_TOOL = {
    "name": "grade_finding",
    "description": "Independently grade whether a security finding is valid given the source code.",
    "input_schema": {
        "type": "object",
        "properties": {
            "is_valid_finding": {
                "type": "boolean",
                "description": "True if this is a genuine issue given the code shown.",
            },
            "severity_appropriate": {
                "type": "boolean",
                "description": "True if the assigned severity matches the actual risk.",
            },
            "reasoning": {"type": "string"},
        },
        "required": ["is_valid_finding", "severity_appropriate", "reasoning"],
    },
}

JUDGE_SYSTEM_PROMPT = """You are an independent security reviewer grading
another reviewer's finding. Be skeptical -- do not assume the original
finding is correct just because it was flagged. Base your grade only on
what the provided source code actually shows."""

def judge_finding(source_code: str, finding: dict) -> dict:
    response = client.messages.create(
        model="claude-sonnet-4-6",  # ideally a different/stronger model than
                                      # whatever generated `finding`, per the
                                      # self-preference bias note above
        max_tokens=1024,
        system=JUDGE_SYSTEM_PROMPT,
        tools=[JUDGE_TOOL],
        tool_choice={"type": "tool", "name": "grade_finding"},
        messages=[{
            "role": "user",
            "content": f"Source code:\n```xpp\n{source_code}\n```\n\n"
                       f"Finding to grade: {finding['title']} "
                       f"(severity: {finding['severity']})\n"
                       f"Description: {finding['description']}",
        }],
    )
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    return tool_use_block.input

# Run this against each of the 3 real runs' authorization-check findings
# from the earlier sessions, and you'd know whether the 2 runs that
# caught it were right to, or whether the 1 run that missed it was
# actually the more defensible call -- independent evidence either way,
# though still not compiler-verified ground truth.

## Interview-ready summary

**Q: "How would you evaluate the quality of an LLM-based system's output without a labeled dataset?"**

Weak answer: "I'd have a human review the outputs."

Strong answer: "Human review doesn't scale and is the actual ground truth I'd want to calibrate against, not replace. LLM-as-judge -- using a separate LLM call to grade outputs against a rubric -- gives a scalable second opinion, but it comes with real, documented biases: self-preference bias if the judge shares a model family with what it's grading, verbosity bias favoring longer outputs, position bias in pairwise comparisons. None of those are theoretical -- they're measurable and need explicit mitigation, like using a different model as judge and swapping comparison order. And critically, the judge's verdict is still a model opinion, not ground truth -- I'd calibrate it against a human-graded sample first to know if it's actually tracking real judgment before trusting it at scale. For a system where the actual ground truth is blocked -- like a code review agent whose findings can't be verified without a compiler -- LLM-as-judge is a genuinely useful interim signal, but I'd be explicit that it narrows the uncertainty, it doesn't eliminate it."